# Bounded agent experiment — existing Colab data

五组开发实验：NO_DATA / ALL_AVAILABLE / RANDOM（5 次）/ AGENT（一次性消融）/
AGENT_ITERATIVE（两步获取）。每模型、每陨石坑四问题共 **36** 条记录。
原 notebook 保留；这个副本没有历史输出或密钥。开发成本仍为 1/2/2 抽象单位，
不是 token 成本。大栅格只读；本 notebook 不下载科学数据，不重新运行 pilot。

1. 先完成私有仓库 Git 授权，再更新代码。若此运行时已导入旧项目模块，重启后运行。
2. 已有正确模型时跳过依赖安装与模型加载；全新运行时按顺序执行。
3. 在「实验配置」修改 CASE_DIR、RESULTS_ROOT、RUN_LABEL；默认 Eminescu 路径沿用旧本。
4. 新 RUN_LABEL 只生成小型视图；旧 TIFF 与实验结果不改动。
5. 两步选择第二次请求后由系统停止；显式 FINISH 与系统停止分别保存。
6. 只比较本轮匹配运行的结果。失败和截断均保留；这不是正式 held-out 实验。


## 1. 更新项目与检查（无需重下载数据）

In [ ]:
import sys

assert sys.version_info >= (3, 12), sys.version

In [ ]:
import subprocess
from pathlib import Path

PROJECT = Path("/content/autonomous-modality-selection")
if (PROJECT / ".git").is_dir():
    dirty = subprocess.check_output(
        ["git", "-C", str(PROJECT), "status", "--porcelain"], text=True
    ).strip()
    if dirty:
        raise RuntimeError("Colab 仓库有未提交改动，请先保存；本单元不会覆盖。")
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "merge", "--ff-only", "origin/main"], check=True)
elif PROJECT.exists():
    raise RuntimeError(f"目录已存在且不是 Git 仓库，请先检查：{PROJECT}")
else:
    # 私有仓库：先完成 Git 认证。不要把访问令牌写进 URL 或 notebook。
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/liaojiqichi/autonomous-modality-selection.git",
            str(PROJECT),
        ],
        check=True,
    )
print(
    "当前提交:",
    subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip(),
)

In [ ]:
%cd /content/autonomous-modality-selection
%pip install -e ".[dev,pilot]"

In [ ]:
!python -m ruff check .
!python -m ruff format --check .
!python -m pytest

In [ ]:
!nvidia-smi

## 2. 模型环境
已加载并可运行时跳过安装；安装后如有提示请重启运行时。

In [ ]:
%pip install -q "transformers>=4.57.1,<5" "accelerate>=1.2,<2" "bitsandbytes>=0.48,<1" pillow

In [ ]:
if "model" in globals():
    if (
        str(globals()["model"].config._name_or_path) != "Qwen/Qwen3-VL-8B-Instruct"
        or "processor" not in globals()
    ):
        raise RuntimeError("已有模型与 Qwen3-VL-8B 不一致，请使用新的运行时。")
    print("复用已加载的 Qwen3-VL-8B；不重复分配显存。")
else:
    import bitsandbytes
    import torch
    import transformers
    from transformers import (
        AutoProcessor,
        BitsAndBytesConfig,
        Qwen3VLForConditionalGeneration,
    )

    assert torch.cuda.is_available(), "请先在 Colab 中选择 GPU 运行时"

    print("GPU:", torch.cuda.get_device_name(0))
    print("PyTorch:", torch.__version__)
    print("Transformers:", transformers.__version__)
    print("bitsandbytes:", bitsandbytes.__version__)

    MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        dtype=torch.float16,
        device_map={"": 0},
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    )
    model.eval()

    processor = AutoProcessor.from_pretrained(MODEL_ID)

    print(
        "模型加载后显存:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GiB",
    )

## 3. 实验配置与小型输入视图
先修改路径和新 RUN_LABEL；旧源数据只读。

In [ ]:
import hashlib
import importlib.metadata
import json
import subprocess
import sys
from pathlib import Path, PureWindowsPath

import numpy as np
import rasterio
import torch
from pydantic import BaseModel, ConfigDict

PROJECT = Path("/content/autonomous-modality-selection")
CASE_DIR = Path("/content/mercury-pilot-001/crater-16604")
RESULTS_ROOT = Path("/content/experiment-results")  # 已挂载 Drive 时可改成 Drive 路径
RUN_LABEL = "emin-agentic-001"  # 新实验名称，不沿用 emin-4conditions-006
BUDGET = 4  # 主实验 4；敏感性实验改为 3
SEED = 42
MAX_MODALITIES = 2
RANDOM_DRAWS = 5
IMAGE_EDGE = 448
ANSWER_TOKENS = 1024
SELECTOR_TOKENS = 256
MAX_INPUT_TOKENS = 8192
RECORD_VERSION = "colab-ap-agentic-development-3"
ITERATIVE_SELECTOR_TOKENS = 512
IS_DEVELOPMENT = True  # 当前 Eminescu 已见过，不允许冒充 held-out

if not IS_DEVELOPMENT or BUDGET not in (3, 4):
    raise ValueError("此 notebook 仅用于开发试验；预算必须为 3 或 4。")
sys.path.insert(0, str(PROJECT / "src"))
sys.path.insert(0, str(PROJECT / "extensions/evidence_views_v1/src"))
from ams_evidence_views.builder import read_grid, render_map, render_profile, terrain_summary

from autonomous_modality.experiments import scenario_seed
from autonomous_modality.pilot import CatalogueRow, PilotCase

MODEL_NAME = str(model.config._name_or_path)
ALLOWED_MODELS = ("Qwen/Qwen3-VL-8B-Instruct", "google/gemma-4-E4B-it")
if MODEL_NAME not in ALLOWED_MODELS:
    raise ValueError(f"实际模型不属于本实验：{MODEL_NAME}")
MODEL_TAG = "qwen3-vl-8b" if ALLOWED_MODELS[0] == MODEL_NAME else "gemma4-e4b"
OUTPUT = RESULTS_ROOT / RUN_LABEL / MODEL_TAG / f"budget-{BUDGET}"


def sha256(path: Path) -> str:
    """分块计算文件哈希，不把大文件一次装入内存。"""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def json_digest(value: object) -> str:
    """用于比较 JSON 配置；拒绝 NaN。"""
    raw = json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False
    )
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def write_new_json(path: Path, value: object) -> None:
    """只创建新文件，不覆盖任何已有结果。"""
    with path.open("x", encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)


def portable_name(value: str) -> str:
    """兼容本地 Windows 生成后上传至 Colab 的路径记录。"""
    return PureWindowsPath(value).name


SOURCES = {
    name: CASE_DIR / name
    for name in (
        "case.json",
        "catalogue.json",
        "image_local.tif",
        "image_context.tif",
        "dem_local.tif",
    )
}
for path in SOURCES.values():
    if not path.is_file():
        raise FileNotFoundError(f"缺少已有数据：{path}；不要重下数据，先检查路径。")
SOURCE_HASHES = {name: sha256(path) for name, path in SOURCES.items()}
case = PilotCase.model_validate_json(SOURCES["case.json"].read_text(encoding="utf-8"))
catalogue = CatalogueRow.model_validate_json(SOURCES["catalogue.json"].read_text(encoding="utf-8"))
if case.catalogue != catalogue or not case.accepted:
    raise ValueError("目录记录与 case.json 不一致，或该案例未通过已有技术检查。")
CASE_ID, CASE_NAME = str(catalogue.id), catalogue.name
products = {portable_name(p.data_path): p for p in case.products}
for name in ("image_local.tif", "image_context.tif", "dem_local.tif"):
    if name not in products or SOURCE_HASHES[name] != products[name].data_sha256:
        raise ValueError(f"源栅格与历史校验值不一致：{name}")
if (
    "metr" not in products["dem_local.tif"].units.lower()
    and "meter" not in products["dem_local.tif"].units.lower()
):
    raise ValueError("DEM 单位不是已记录的米，不能直接生成数值摘要。")

question_path = PROJECT / "configs/mercury_questions_v1.json"
QUESTIONS = json.loads(question_path.read_text(encoding="utf-8"))["questions"]
if [q["question_id"] for q in QUESTIONS] != ["Q1", "Q2", "Q3", "Q4"]:
    raise ValueError("本轮只使用四个已有开发问题。")


class ViewReceipt(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    version: str
    source_sha256: dict[str, str]
    implementation_sha256: dict[str, str]
    image_edge: int
    output_sha256: dict[str, str]


view_code = PROJECT / "extensions/evidence_views_v1/src/ams_evidence_views"
CODE_HASHES = {
    str(p.relative_to(PROJECT)): sha256(p)
    for root in (PROJECT / "src/autonomous_modality", view_code)
    for p in sorted(root.glob("*.py"))
}
# 小型模型输入文件；不复制 TIFF，不改原案例。
VIEW_DIR = RESULTS_ROOT / RUN_LABEL / "shared_inputs" / CASE_ID
PREVIEWS = {
    "optical_local": VIEW_DIR / "optical_local.png",
    "optical_context": VIEW_DIR / "optical_context.png",
    "elevation": VIEW_DIR / "elevation.png",
    "profile_east": VIEW_DIR / "profile_east.png",
    "profile_north": VIEW_DIR / "profile_north.png",
}
view_spec = {
    "version": "notebook-pilot-views-1",
    "source_sha256": SOURCE_HASHES,
    "implementation_sha256": CODE_HASHES,
    "image_edge": IMAGE_EDGE,
}
receipt_path = VIEW_DIR / "receipt.json"
if VIEW_DIR.exists():
    if not receipt_path.is_file():
        raise RuntimeError("发现不完整输入目录；请使用新的 RUN_LABEL，不自动覆盖。")
    receipt = ViewReceipt.model_validate_json(receipt_path.read_text(encoding="utf-8"))
    if receipt.model_dump(exclude={"output_sha256"}) != view_spec:
        raise ValueError("源数据或表示代码变化，请使用新的 RUN_LABEL。")
    expected_files = {p.name for p in PREVIEWS.values()} | {"terrain.txt"}
    if set(receipt.output_sha256) != expected_files:
        raise ValueError("输入表示清单不完整。")
    for name, digest in receipt.output_sha256.items():
        if sha256(VIEW_DIR / name) != digest:
            raise ValueError(f"输入文件被修改：{name}")
else:
    if OUTPUT.exists() and any(OUTPUT.iterdir()):
        raise ValueError("输出目录已有其他内容；请使用新的 RUN_LABEL。")
    VIEW_DIR.mkdir(parents=True, exist_ok=False)
    grids = {}
    for name in ("image_local.tif", "image_context.tif", "dem_local.tif"):
        grids[name] = read_grid(SOURCES[name])
        crs = rasterio.crs.CRS.from_wkt(grids[name][1]["crs_wkt"]).to_dict()
        if (
            crs.get("proj") != "aeqd"
            or not np.isclose(crs.get("lat_0", 999), catalogue.lat_n)
            or not np.isclose(crs.get("lon_0", 999), catalogue.lon_e_0)
        ):
            raise ValueError(f"栅格中心与目录不符：{name}")
    for stem, name in (
        ("optical_local", "image_local.tif"),
        ("optical_context", "image_context.tif"),
        ("elevation", "dem_local.tif"),
    ):
        render_map(
            PREVIEWS[stem], *grids[name], f"{CASE_NAME}: {stem}", IMAGE_EDGE, stem == "elevation"
        )
    summary = terrain_summary(*grids["dem_local.tif"], products["dem_local.tif"].units)
    sampled = summary.model_dump(mode="json")
    for profile in summary.profiles:
        render_profile(PREVIEWS[f"profile_{profile.axis}"], profile, IMAGE_EDGE)
    for profile in sampled["profiles"]:
        indices = np.unique(np.linspace(0, len(profile["distance_km"]) - 1, 17, dtype=int))
        for key in ("distance_km", "elevation_m"):
            profile[key] = [profile[key][int(i)] for i in indices]
    terrain_text = (
        "Whole-crop statistics, NOT crater depth. Fixed central east/north profiles; "
        "even dimensions average the middle two lines where both are valid. "
        "17 index-spaced samples per profile need not preserve extrema. "
        "Null means missing. Optical and DEM share MDIS ancestry. "
        "Numeric TIFF/CSV files are not directly read by the answer model.\n"
        + json.dumps(sampled, ensure_ascii=False, allow_nan=False)
    )
    (VIEW_DIR / "terrain.txt").write_text(terrain_text, encoding="utf-8")
    if {name: sha256(path) for name, path in SOURCES.items()} != SOURCE_HASHES:
        raise ValueError("处理过程中源文件变化；当前输入包不完整，请检查。")
    receipt = ViewReceipt(
        **view_spec,
        output_sha256={p.name: sha256(p) for p in [*PREVIEWS.values(), VIEW_DIR / "terrain.txt"]},
    )
    write_new_json(receipt_path, receipt.model_dump())
TERRAIN_TEXT = (VIEW_DIR / "terrain.txt").read_text(encoding="utf-8")
CATALOGUE_TEXT = json.dumps(catalogue.model_dump(), ensure_ascii=False)
OUTPUT.mkdir(parents=True, exist_ok=True)

ASSETS = {
    "CRATER_CATALOG": {
        "cost": 1,
        "description": (
            "Single historical target record; not a regional population or ground truth."
        ),
    },
    "OPTICAL_IMAGE": {
        "cost": 2,
        "description": (
            "Local and regional optical views; stretched DN, not composition or absolute age."
        ),
    },
    "TOPOGRAPHY": {
        "cost": 2,
        "description": (
            "Elevation map, fixed east/north profile plots, crop statistics "
            "and sampled numeric values in metres; not a measured crater depth. "
            "Raster itself is not directly analyzed by the LLM."
        ),
    },
}
SELECTOR_TASK = (
    "Choose one supplied feasible input combination expected to support distinct, "
    "relevant analytical approaches and explanatory perspectives."
)
ANSWER_POLICY = (
    "Assist an exploratory study of a Mercury crater. "
    "Propose distinct, concrete, executable analytical approaches and relevant explanatory "
    "mechanisms or alternatives. Do not add routine steps or paraphrases just to increase counts. "
    "Separate proposed analysis, hypotheses, observations and analyses actually performed. "
    "Do not claim unperformed measurements or invent observations. "
    "Crosses, axes, legends and profile markers are annotations, not geological features. "
    "The optical mosaic and DEM share MDIS ancestry; "
    "agreement is not independent sensor confirmation. "
    "A single catalog record does not support population correlations. "
    "Without data, propose relevant investigations without claiming observed features. "
    "Answer in Chinese, aiming for 400–600 Chinese characters; no required number of ideas."
)
CONFIG = {
    "protocol": RECORD_VERSION,
    "split": "development",
    "case_id": CASE_ID,
    "case_name": CASE_NAME,
    "model": MODEL_NAME,
    "model_revision": getattr(model.config, "_commit_hash", None),
    "four_bit": bool(getattr(model, "is_loaded_in_4bit", False)),
    "seed": SEED,
    "budget": BUDGET,
    "maximum_modalities": MAX_MODALITIES,
    "random_draws": RANDOM_DRAWS,
    "image_edge": IMAGE_EDGE,
    "answer_tokens": ANSWER_TOKENS,
    "selector_tokens": SELECTOR_TOKENS,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "decoding": "greedy_one_beam",
    "thinking": "disabled",
    "cost_unit": "legacy_ordinal_units",
    "iterative_selector_tokens": ITERATIVE_SELECTOR_TOKENS,
    "question_sha256": sha256(question_path),
    "source_sha256": SOURCE_HASHES,
    "view_receipt": receipt.model_dump(),
    "selector_task": SELECTOR_TASK,
    "answer_policy": ANSWER_POLICY,
    "assets": ASSETS,
    "repository_commit": subprocess.check_output(
        ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
    ).strip(),
    "implementation_sha256": CODE_HASHES,
}
print("开发案例:", CASE_NAME, "| 模型:", MODEL_NAME, "| 输出:", OUTPUT)
print("每个模型：4 个问题 × (1 + 1 + 5 + 1 + 1) = 36 条开发记录。")

## 4. 推理统计、旧选择接口与统一回答接口

In [ ]:
import copy
import time
from typing import Any, Literal

import torch
from pydantic import BaseModel, ConfigDict, Field


class GenerationTrace(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True, allow_inf_nan=False)
    messages: list[dict[str, Any]]
    max_new_tokens: int
    input_tokens: int | None = None
    output_tokens: int | None = None
    finish_reason: Literal["error", "eos", "length", "unknown"] = "error"
    elapsed_seconds: float = 0.0
    peak_gpu_gib: float | None = None
    text: str | None = None
    error: str | None = None


GENERATION_LOG = []


def generate_reply(messages: list[dict], max_new_tokens: int = 128) -> str:
    """接口不变；保存每次真实调用的结果、错误和资源指标。"""
    if max_new_tokens < 1:
        raise ValueError("max_new_tokens 必须为正数。")
    trace = GenerationTrace(messages=copy.deepcopy(messages), max_new_tokens=max_new_tokens)
    record = trace.model_dump()
    GENERATION_LOG.append(record)
    inputs = outputs = new_tokens = None
    started = time.perf_counter()
    try:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        template_kwargs = {
            "tokenize": True,
            "add_generation_prompt": True,
            "return_dict": True,
            "return_tensors": "pt",
        }

        # 当前 Qwen3-VL-Instruct processor 不接受这个参数。
        # 仅在 Gemma 分支传入。
        if MODEL_NAME == "google/gemma-4-E4B-it":
            template_kwargs["enable_thinking"] = False

        inputs = processor.apply_chat_template(
            messages,
            **template_kwargs,
        ).to("cuda:0", dtype=model.dtype)
        n_input = int(inputs["input_ids"].shape[-1])
        trace.input_tokens = n_input
        if n_input > MAX_INPUT_TOKENS:
            raise ValueError(f"INPUT_TOKEN_LIMIT: {n_input} > {MAX_INPUT_TOKENS}")
        has_images = any(
            b.get("type") == "image" for message in messages for b in message["content"]
        )
        if has_images and not any("pixel" in name for name in inputs):
            raise ValueError("图像没有进入 processor 输出，禁止当作多模态成功运行。")
        config = copy.deepcopy(model.generation_config)
        config.do_sample = False
        config.num_beams = 1
        config.temperature = None
        config.top_p = None
        config.top_k = None
        config.max_new_tokens = max_new_tokens
        config.use_cache = True
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                generation_config=config,
                use_model_defaults=False,
                do_sample=False,
                num_beams=1,
                max_new_tokens=max_new_tokens,
                use_cache=True,
                return_dict_in_generate=False,
            )
        new_tokens = outputs[:, n_input:]
        token_ids = new_tokens[0].tolist()
        trace.text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
        trace.output_tokens = len(token_ids)
        eos = config.eos_token_id
        if eos is None:
            eos = processor.tokenizer.eos_token_id
        eos_ids = set(eos if isinstance(eos, (list, tuple)) else [eos])
        trace.finish_reason = (
            "eos"
            if token_ids and token_ids[-1] in eos_ids
            else "length"
            if len(token_ids) >= max_new_tokens
            else "unknown"
        )
        torch.cuda.synchronize()
        return trace.text
    except Exception as exc:
        trace.error = f"{type(exc).__name__}: {exc}"
        raise
    finally:
        trace.elapsed_seconds = time.perf_counter() - started
        trace.peak_gpu_gib = torch.cuda.max_memory_allocated() / 1024**3
        record.update(trace.model_dump())
        del inputs, outputs, new_tokens

In [ ]:
import random
from itertools import combinations

from pydantic import BaseModel, ConfigDict


class AgentChoice(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    option_id: str
    rationale: str = Field(min_length=1)


def valid_selection(selected: object) -> list[str]:
    """仅用于 RANDOM/AGENT；ALL_AVAILABLE 是不受选模预算限制的参照。"""
    if not isinstance(selected, list) or not all(isinstance(x, str) for x in selected):
        raise ValueError("选择必须是字符串列表。")
    if not 1 <= len(selected) <= MAX_MODALITIES or len(set(selected)) != len(selected):
        raise ValueError("选择数量或重复模态不合法。")
    if set(selected) - set(ASSETS):
        raise ValueError("未知模态。")
    if sum(ASSETS[x]["cost"] for x in selected) > BUDGET:
        raise ValueError("选择超出预算。")
    return sorted(selected)


FEASIBLE = [
    list(items)
    for n in range(1, MAX_MODALITIES + 1)
    for items in combinations(sorted(ASSETS), n)
    if sum(ASSETS[x]["cost"] for x in items) <= BUDGET
]
OPTIONS = {
    f"S{i + 1}": {"selected": items, "total_cost": sum(ASSETS[x]["cost"] for x in items)}
    for i, items in enumerate(FEASIBLE)
}
if not OPTIONS:
    raise ValueError("NO_FEASIBLE_OPTIONS")


def select_inputs(
    condition: str, question: str, question_id: str, draw_index: int = 0
) -> tuple[list[str], str]:
    """保留原三个位置参数；新增可选 draw_index，不依赖原函数备份。"""
    if type(draw_index) is not int or not 0 <= draw_index < RANDOM_DRAWS:
        raise ValueError("非法 draw_index。")
    if condition != "RANDOM" and draw_index != 0:
        raise ValueError("非随机条件只运行一次。")
    if condition == "NO_DATA":
        return [], "No additional scientific evidence."
    if condition == "ALL_AVAILABLE":
        return sorted(ASSETS), "All available modalities; non-budget-matched reference."
    if condition == "RANDOM":
        seed = scenario_seed(SEED, f"{CASE_ID}-{question_id}", BUDGET, draw_index)
        return valid_selection(random.Random(seed).choice(FEASIBLE)), (
            "Uniform feasible-subset draw with replacement."
        )
    if condition != "AGENT":
        raise ValueError(f"Unknown condition: {condition}")

    instruction = {
        "task": SELECTOR_TASK,
        "question": question,
        "modality_descriptions": ASSETS,
        "feasible_options": OPTIONS,
        "rules": [
            "Choose exactly one supplied option_id; do not change the option.",
            "Budget is a ceiling; spending less receives no separate reward.",
            "The catalog is one historical target record, not population data.",
            "Inputs include plotted views and a sampled numeric terrain summary, "
            "not executable rasters.",
            "Acknowledge representation limitations; "
            "focus on analytical approaches and explanations.",
            "Return only JSON with exactly option_id and a nonempty rationale.",
        ],
    }
    raw = generate_reply(
        [
            {
                "role": "user",
                "content": [{"type": "text", "text": json.dumps(instruction, ensure_ascii=False)}],
            }
        ],
        max_new_tokens=SELECTOR_TOKENS,
    )
    # 原文已保存在 GENERATION_LOG，运行单元在成功和失败时均保存。
    if GENERATION_LOG[-1]["finish_reason"] != "eos":
        raise ValueError("SELECTOR_OUTPUT_NOT_COMPLETE")
    parsed = AgentChoice.model_validate_json(raw)
    if parsed.option_id not in OPTIONS or not parsed.rationale.strip():
        raise ValueError("INVALID_AGENT_CHOICE")
    return valid_selection(OPTIONS[parsed.option_id]["selected"]), parsed.rationale


assert len(FEASIBLE) == (6 if BUDGET == 4 else 5)
assert select_inputs("NO_DATA", "test", "Q1") == ([], "No additional scientific evidence.")
print("可行组合:", OPTIONS)
print(
    "Q1 五次随机抽样（重复组合正常）:",
    [select_inputs("RANDOM", "test", "Q1", i)[0] for i in range(RANDOM_DRAWS)],
)

In [ ]:
def build_answer_messages(
    question: str, requirements: list[str], selected: list[str]
) -> list[dict]:
    """回答模型只得到选中证据；不传条件名、未选模态说明或 selector rationale。"""
    if len(set(selected)) != len(selected) or set(selected) - set(ASSETS):
        raise ValueError("非法证据选择。")
    prompt = (
        ANSWER_POLICY
        + f"\nTarget: {CASE_NAME}\nQuestion:\n{question}\nRequirements:\n"
        + "\n".join(f"- {x}" for x in requirements)
    )
    images, texts = [], []
    if "CRATER_CATALOG" in selected:
        texts.append(
            "Historical catalogue record; morphology codes are not ground truth. "
            "Coordinates: degrees; diameter: km. "
            "int_shp: b=bowl, sh=slump hummocks, ff=flat floor; "
            "rim_shp: c=circular, sc=scalloped, t=terraced; "
            "cent_struc: cp=central peak, mp=multiple peaks, pi=central pit, "
            "pr=peak ring, mr=multi-ring, n=none; x=poorly imaged; "
            "rayed: y=yes, n=no. Unknown codes must not be guessed.\n" + CATALOGUE_TEXT
        )
    if "OPTICAL_IMAGE" in selected:
        for key in ("optical_local", "optical_context"):
            images.append({"type": "image", "image": str(PREVIEWS[key])})
            texts.append(f"Image {len(images)}: {key}; display-stretched optical view.")
    if "TOPOGRAPHY" in selected:
        for key in ("elevation", "profile_east", "profile_north"):
            images.append({"type": "image", "image": str(PREVIEWS[key])})
            texts.append(f"Image {len(images)}: {key}; elevation in metres, not crater depth.")
        texts.append(TERRAIN_TEXT)
    if not selected:
        texts.append("No additional scientific evidence is supplied.")
    # 两个模型使用同一内容和顺序，图像在前，文字说明在后。
    content = [*images, {"type": "text", "text": prompt + "\n\n" + "\n\n".join(texts)}]
    return [{"role": "user", "content": content}]


def answer_question(question: str, requirements: list[str], selected: list[str]) -> str:
    """调用当前加载模型，token 上限由运行配置记录。"""
    return generate_reply(
        build_answer_messages(question, requirements, selected), max_new_tokens=ANSWER_TOKENS
    )


# 只检查路由，不调用模型。
for chosen, expected in [
    ([], 0),
    (["CRATER_CATALOG"], 0),
    (["OPTICAL_IMAGE"], 2),
    (["TOPOGRAPHY"], 3),
    (sorted(ASSETS), 5),
]:
    content = build_answer_messages("fixture", [], chosen)[0]["content"]
    assert sum(x["type"] == "image" for x in content) == expected
print("输入路由检查通过：0/0/2/3/5 张图。")

## 5. 两步 agent 与 receipt 输入包兼容层
此单元仅定义接口和校验输入。

In [ ]:
from autonomous_modality.iterative import (
    ContentBlock,
    EvidenceObservation,
    IterativePolicy,
    IterativeSelection,
    run_iterative_selection,
)
from autonomous_modality.iterative_colab import notebook_generator
from autonomous_modality.models import (
    CraterQuestion,
    CraterReference,
    DataAssetProfile,
    InputDataModality,
    InputSelectionConstraints,
    InputSelectionRequest,
)

ITERATIVE_POLICY = IterativePolicy(
    cost_unit="legacy_ordinal_units",
    cost_definition="Development pilot: catalogue=1, optical=2, topography=2; complete packages.",
    selector_max_new_tokens=ITERATIVE_SELECTOR_TOKENS,
)
PINNED_RECEIPT_SHA256 = sha256(receipt_path)


def build_evidence_content(selected: list[str]) -> list[dict]:
    """Reuse the original routing, removing only the shared answer instruction prefix."""
    question = "Evidence routing placeholder"
    prefix = (
        ANSWER_POLICY + f"\nTarget: {CASE_NAME}\nQuestion:\n{question}\nRequirements:\n" + "\n\n"
    )
    content = copy.deepcopy(build_answer_messages(question, [], selected)[0]["content"])
    if content[-1]["type"] != "text" or not content[-1]["text"].startswith(prefix):
        raise ValueError("Answer routing changed; verify evidence-only extraction.")
    content[-1]["text"] = content[-1]["text"][len(prefix) :]
    return content


def load_notebook_evidence(modality: InputDataModality) -> EvidenceObservation:
    """Return actual receipt-verified input bytes through the original notebook routing."""
    if sha256(receipt_path) != PINNED_RECEIPT_SHA256:
        raise ValueError("Input receipt changed during acquisition.")
    for name, digest in receipt.output_sha256.items():
        if sha256(VIEW_DIR / name) != digest:
            raise ValueError(f"Input view changed: {name}")
    if sha256(SOURCES["catalogue.json"]) != SOURCE_HASHES["catalogue.json"]:
        raise ValueError("Catalogue changed during acquisition.")
    actual_catalogue = CatalogueRow.model_validate_json(
        SOURCES["catalogue.json"].read_text(encoding="utf-8")
    )
    if json.dumps(actual_catalogue.model_dump(), ensure_ascii=False) != CATALOGUE_TEXT:
        raise ValueError("In-memory catalogue differs from pinned input.")
    if (VIEW_DIR / "terrain.txt").read_text(encoding="utf-8") != TERRAIN_TEXT:
        raise ValueError("In-memory terrain differs from pinned input.")
    files_by_modality = {
        "CRATER_CATALOG": {"catalogue.json": SOURCE_HASHES["catalogue.json"]},
        "OPTICAL_IMAGE": {
            name: receipt.output_sha256[name]
            for name in ("optical_local.png", "optical_context.png")
        },
        "TOPOGRAPHY": {
            name: receipt.output_sha256[name]
            for name in ("elevation.png", "profile_east.png", "profile_north.png", "terrain.txt")
        },
    }
    return EvidenceObservation(
        modality=modality,
        blocks=[ContentBlock.model_validate(b) for b in build_evidence_content([modality.value])],
        evidence_sha256=files_by_modality[modality.value],
        package_sha256=PINNED_RECEIPT_SHA256,
    )


def select_inputs_iterative(question: str, question_id: str) -> IterativeSelection:
    """A separate entry point; the original select_inputs function remains unchanged."""
    spec = next(q for q in QUESTIONS if q["question_id"] == question_id)
    request = InputSelectionRequest(
        question=CraterQuestion(
            question_id=f"{CASE_ID}-{question_id}",
            text=question,
            question_type=spec["question_type"],
            crater=CraterReference(
                crater_id=CASE_ID,
                name=CASE_NAME,
                latitude=catalogue.lat_n,
                longitude=catalogue.lon_e_0,
            ),
        ),
        assets=[
            DataAssetProfile(
                asset_id=name,
                modality=name,
                title=name,
                estimated_cost=asset["cost"],
                limitations=[asset["description"]],
            )
            for name, asset in sorted(ASSETS.items())
        ],
        constraints=InputSelectionConstraints(
            maximum_modalities=MAX_MODALITIES,
            maximum_total_cost=BUDGET,
        ),
    )
    return run_iterative_selection(
        request,
        ITERATIVE_POLICY,
        notebook_generator(generate_reply, GENERATION_LOG),
        load_notebook_evidence,
        model_id=MODEL_NAME,
        execution_kind="model",
    )


# Routing checks only: no model call. The final answer still uses answer_question
# for ALL five conditions, preserving the exact input ordering and answer prompt.
for modality, images in [("CRATER_CATALOG", 0), ("OPTICAL_IMAGE", 2), ("TOPOGRAPHY", 3)]:
    observation = load_notebook_evidence(InputDataModality(modality))
    assert sum(b.type == "image" for b in observation.blocks) == images
    assert all(ANSWER_POLICY not in (b.text or "") for b in observation.blocks)
print("两步选择接口就绪；真实输入路由已校验，未调用模型。")

## 6. 执行五组实验
此单元开始 GPU 推理；原 AGENT 保留为消融。

In [ ]:
import gc
import inspect
import traceback
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, model_validator

from autonomous_modality.protocol import ExperimentProtocol

CONDITIONS = ["NO_DATA", "ALL_AVAILABLE", "RANDOM", "AGENT", "AGENT_ITERATIVE"]


class Attempt(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True, allow_inf_nan=False)
    record_version: Literal["colab-ap-agentic-development-3"] = "colab-ap-agentic-development-3"
    run_sha256: str
    model: str
    case_id: str
    question_id: str
    question: str
    condition: Literal["NO_DATA", "ALL_AVAILABLE", "RANDOM", "AGENT", "AGENT_ITERATIVE"]
    draw_index: int = Field(ge=0, le=4)
    selection_seed: int | None = None
    status: Literal["success", "truncated", "incomplete", "error"]
    selected: list[str]
    selection_rationale: str
    iterative_selection: IterativeSelection | None = None
    input_cost: int = Field(ge=0)
    elapsed_seconds: float = Field(ge=0)
    selection_seconds: float = Field(ge=0)
    answer_seconds: float = Field(ge=0)
    peak_gpu_gib: float | None = None
    selection_generations: list[GenerationTrace]
    answer_generations: list[GenerationTrace]
    answer: str | None = None
    finish_reason: Literal["eos", "length", "unknown"] | None = None
    error_stage: Literal["selection", "answer"] | None = None
    error_type: str | None = None
    error: str | None = None
    traceback: str | None = None

    @model_validator(mode="after")
    def check_record(self) -> "Attempt":
        if self.condition != "RANDOM" and self.draw_index != 0:
            raise ValueError("非随机条件不能包含额外重复。")
        if self.condition == "NO_DATA" and self.selected:
            raise ValueError("NO_DATA 不得携带数据。")
        if set(self.selected) - set(ASSETS) or len(set(self.selected)) != len(self.selected):
            raise ValueError("记录中包含非法模态。")
        if self.input_cost != sum(ASSETS[x]["cost"] for x in self.selected):
            raise ValueError("成本不一致。")
        if self.selected and self.condition in {"RANDOM", "AGENT", "AGENT_ITERATIVE"}:
            valid_selection(self.selected)
        if self.status != "error":
            if self.condition == "ALL_AVAILABLE" and self.selected != sorted(ASSETS):
                raise ValueError("ALL_AVAILABLE 缺少模态。")
            if self.condition not in {"NO_DATA", "AGENT_ITERATIVE"} and not self.selected:
                raise ValueError("非空数据条件缺少选择。")
            if not self.answer or not self.answer_generations:
                raise ValueError("回答内容或调用记录缺失。")
            if self.status == "success" and self.finish_reason != "eos":
                raise ValueError("未正常结束的回答不能标为成功。")
        if self.condition == "AGENT_ITERATIVE":
            if self.status != "error" and self.iterative_selection is None:
                raise ValueError("Missing iterative acquisition trace.")
            if self.iterative_selection is not None:
                trace = self.iterative_selection
                if sorted(m.value for m in trace.accessed_modalities) != self.selected:
                    raise ValueError("Iterative access history differs from selected evidence.")
                if trace.cumulative_cost != self.input_cost:
                    raise ValueError("Iterative cumulative cost mismatch.")
                if self.status != "error" and trace.status != "ready":
                    raise ValueError("Cannot answer after failed acquisition.")
        elif self.iterative_selection is not None:
            raise ValueError("Iterative trace on another condition.")
        return self


protocol_path = PROJECT / "configs/experiment_protocol.json"
protocol = ExperimentProtocol.model_validate_json(protocol_path.read_text(encoding="utf-8"))
live_settings = {
    "model": str(model.config._name_or_path),
    "seed": SEED,
    "budget": BUDGET,
    "maximum_modalities": MAX_MODALITIES,
    "random_draws": RANDOM_DRAWS,
    "image_edge": IMAGE_EDGE,
    "answer_tokens": ANSWER_TOKENS,
    "selector_tokens": SELECTOR_TOKENS,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "iterative_selector_tokens": ITERATIVE_SELECTOR_TOKENS,
}
if any(CONFIG[key] != value for key, value in live_settings.items()):
    raise ValueError("变量与已构造配置不一致，请从 02 开始按顺序执行。")
if MODEL_NAME not in protocol.models:
    raise ValueError("模型与项目协议不一致。")
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# inspect.getsource 在正常 Colab 单元中可用；失败时明确报错，不静默省略代码指纹。
tracked_objects = (
    sha256,
    json_digest,
    write_new_json,
    portable_name,
    generate_reply,
    valid_selection,
    select_inputs,
    build_answer_messages,
    answer_question,
    Attempt.check_record,
    build_evidence_content,
    load_notebook_evidence,
    select_inputs_iterative,
)
try:
    function_sources = {x.__name__: inspect.getsource(x) for x in tracked_objects}
except (OSError, TypeError) as exc:
    raise RuntimeError("无法获取单元源代码；请按顺序重新执行替换单元。") from exc

metadata = {
    "config": CONFIG,
    "protocol": protocol.model_dump(mode="json"),
    "protocol_sha256": sha256(protocol_path),
    "functions": function_sources,
    "iterative_policy": ITERATIVE_POLICY.model_dump(mode="json"),
    "condition_roles": {"AGENT": "one_shot_ablation", "AGENT_ITERATIVE": "primary_agent"},
    "schemas": {
        cls.__name__: cls.model_json_schema()
        for cls in (GenerationTrace, AgentChoice, Attempt, ViewReceipt, IterativeSelection)
    },
    "options": OPTIONS,
    "model_config": json.loads(model.config.to_json_string()),
    "generation_defaults": json.loads(model.generation_config.to_json_string()),
    "processor_class": type(processor).__name__,
    "image_processor": json.loads(processor.image_processor.to_json_string()),
    "chat_template": getattr(processor, "chat_template", None),
    "tokenizer_chat_template": getattr(processor.tokenizer, "chat_template", None),
    "tokenizer_class": type(processor.tokenizer).__name__,
    "environment": {
        name: importlib.metadata.version(name)
        for name in (
            "torch",
            "transformers",
            "bitsandbytes",
            "accelerate",
            "numpy",
            "Pillow",
            "rasterio",
            "matplotlib",
            "pydantic",
        )
    },
    "gpu": torch.cuda.get_device_name(0),
    "cuda": torch.version.cuda,
    "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
}
# 重新核对实际输入，不允许在原 RUN_LABEL 下悄悄变更。
if {name: sha256(path) for name, path in SOURCES.items()} != SOURCE_HASHES:
    raise ValueError("源文件已变化。")
for name, digest in receipt.output_sha256.items():
    if sha256(VIEW_DIR / name) != digest:
        raise ValueError(f"输入副本已变化：{name}")
# 同一 RUN_LABEL 下两模型共享问题、输入文件字节、提示词和随机组合。
# 若跨会话运行，请使用持久化 RESULTS_ROOT，或先恢复之前下载的实验目录。
paired_design = {
    "case_id": CASE_ID,
    "source_sha256": SOURCE_HASHES,
    "view_sha256": receipt.output_sha256,
    "question_sha256": CONFIG["question_sha256"],
    "budget": BUDGET,
    "maximum_modalities": MAX_MODALITIES,
    "answer_tokens": ANSWER_TOKENS,
    "selector_tokens": SELECTOR_TOKENS,
    "answer_policy": ANSWER_POLICY,
    "selector_task": SELECTOR_TASK,
    "options": OPTIONS,
    "iterative_policy": ITERATIVE_POLICY.model_dump(mode="json"),
    "conditions": CONDITIONS,
    "random_choices": {
        q["question_id"]: [
            select_inputs("RANDOM", q["text"], q["question_id"], i)[0] for i in range(RANDOM_DRAWS)
        ]
        for q in QUESTIONS
    },
}
paired_path = RESULTS_ROOT / RUN_LABEL / f"paired-{CASE_ID}-b{BUDGET}.json"
if paired_path.exists():
    if json.loads(paired_path.read_text(encoding="utf-8")) != paired_design:
        raise ValueError("两模型的共享设计或实际输入不一致，禁止混合比较。")
else:
    write_new_json(paired_path, paired_design)
metadata["paired_design_sha256"] = sha256(paired_path)
metadata_path = OUTPUT / "run_metadata.json"
if metadata_path.exists():
    if json.loads(metadata_path.read_text(encoding="utf-8")) != metadata:
        raise ValueError("代码、模型或配置变化；使用新的 RUN_LABEL。")
else:
    if list(OUTPUT.glob("Q*__*.json")):
        raise ValueError("存在无对应配置的旧结果；使用新的 RUN_LABEL。")
    write_new_json(metadata_path, metadata)
RUN_SHA256 = sha256(metadata_path)

# 最重输入的短 smoke test，只检查图文接入；不保证 1024-token 完整回答一定不会 OOM。
# 如失败，停止；不能自动删图、缩小某一组输入或用规则替换 agent。
preflight_path = OUTPUT / "preflight.json"
if not preflight_path.exists():
    GENERATION_LOG.clear()
    try:
        generate_reply(
            build_answer_messages(
                "请简要说明可以从这些输入开展什么研究，不作定量结论。", [], sorted(ASSETS)
            ),
            max_new_tokens=32,
        )
        preflight = {
            "run_sha256": RUN_SHA256,
            "status": "completed_input_smoke_test",
            "generation": GENERATION_LOG[-1],
        }
    except Exception:
        write_new_json(
            preflight_path,
            {
                "run_sha256": RUN_SHA256,
                "status": "error",
                "generation": GENERATION_LOG[-1] if GENERATION_LOG else None,
            },
        )
        raise
    write_new_json(preflight_path, preflight)
else:
    preflight = json.loads(preflight_path.read_text(encoding="utf-8"))
    if preflight.get("run_sha256") != RUN_SHA256 or preflight.get("status") == "error":
        raise ValueError("预检配置不一致或此前预检失败；处理后使用新的 RUN_LABEL。")

for spec in QUESTIONS:
    question_id = spec["question_id"]
    question = spec["text"].replace("{crater}", CASE_NAME)
    for condition in CONDITIONS:
        for draw_index in range(RANDOM_DRAWS if condition == "RANDOM" else 1):
            destination = OUTPUT / f"{question_id}__{condition}__r{draw_index + 1:02d}.json"
            selection_seed = (
                scenario_seed(SEED, f"{CASE_ID}-{question_id}", BUDGET, draw_index)
                if condition == "RANDOM"
                else None
            )
            if destination.exists():
                old = Attempt.model_validate_json(destination.read_text(encoding="utf-8"))
                actual = (
                    old.run_sha256,
                    old.model,
                    old.case_id,
                    old.question_id,
                    old.condition,
                    old.draw_index,
                    old.selection_seed,
                )
                expected = (
                    RUN_SHA256,
                    MODEL_NAME,
                    CASE_ID,
                    question_id,
                    condition,
                    draw_index,
                    selection_seed,
                )
                if actual != expected:
                    raise ValueError(f"已有结果身份不符：{destination.name}")
                print("保留已有尝试（包括失败）:", destination.name, old.status)
                continue

            GENERATION_LOG.clear()
            started = stage_started = time.perf_counter()
            stage = "selection"
            selected, rationale = [], ""
            iterative_selection = None
            selection_seconds = answer_seconds = 0.0
            selection_generations, answer_generations = [], []
            outcome = {}
            try:
                if condition == "AGENT_ITERATIVE":
                    iterative_selection = select_inputs_iterative(question, question_id)
                    selected = sorted(m.value for m in iterative_selection.accessed_modalities)
                    rationale = " | ".join(
                        t.action.reason for t in iterative_selection.turns if t.action is not None
                    )
                    if iterative_selection.status != "ready":
                        raise ValueError(
                            f"{iterative_selection.stop_reason}: {iterative_selection.error}"
                        )
                else:
                    selected, rationale = select_inputs(
                        condition, question, question_id, draw_index=draw_index
                    )
                selection_seconds = time.perf_counter() - stage_started
                selection_generations = copy.deepcopy(GENERATION_LOG)
                GENERATION_LOG.clear()
                stage, stage_started = "answer", time.perf_counter()
                if iterative_selection is not None:
                    for turn in iterative_selection.turns:
                        if turn.observation is not None:
                            current = load_notebook_evidence(turn.observation.modality)
                            if current != turn.observation:
                                raise ValueError("Acquired evidence changed before answering.")
                answer = answer_question(question, spec["answer_requirements"], selected)
                answer_seconds = time.perf_counter() - stage_started
                answer_generations = copy.deepcopy(GENERATION_LOG)
                if not answer_generations or not answer.strip():
                    raise ValueError("EMPTY_OR_UNLOGGED_ANSWER")
                finish = answer_generations[-1]["finish_reason"]
                outcome = {
                    "status": {"eos": "success", "length": "truncated"}.get(finish, "incomplete"),
                    "answer": answer,
                    "finish_reason": finish,
                }
            except Exception as exc:
                if stage == "selection":
                    selection_seconds = time.perf_counter() - stage_started
                    selection_generations = copy.deepcopy(GENERATION_LOG)
                else:
                    answer_seconds = time.perf_counter() - stage_started
                    answer_generations = copy.deepcopy(GENERATION_LOG)
                outcome = {
                    "status": "error",
                    "error_stage": stage,
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                    "traceback": traceback.format_exc(),
                }
            peaks = [
                g["peak_gpu_gib"]
                for g in selection_generations + answer_generations
                if g.get("peak_gpu_gib") is not None
            ]
            result = Attempt(
                run_sha256=RUN_SHA256,
                model=MODEL_NAME,
                case_id=CASE_ID,
                question_id=question_id,
                question=question,
                condition=condition,
                draw_index=draw_index,
                selection_seed=selection_seed,
                selected=selected,
                selection_rationale=rationale,
                iterative_selection=iterative_selection,
                input_cost=sum(ASSETS[x]["cost"] for x in selected),
                elapsed_seconds=time.perf_counter() - started,
                selection_seconds=selection_seconds,
                answer_seconds=answer_seconds,
                peak_gpu_gib=max(peaks, default=None),
                selection_generations=selection_generations,
                answer_generations=answer_generations,
                **outcome,
            )
            temporary = destination.with_suffix(".json.tmp")
            write_new_json(temporary, result.model_dump(mode="json"))
            if destination.exists():
                raise RuntimeError("请勿同时启动两个写入相同 RUN_LABEL 的 notebook。")
            temporary.replace(destination)
            print(
                question_id,
                condition,
                draw_index + 1,
                result.status,
                selected,
                result.finish_reason or result.error,
            )
            GENERATION_LOG.clear()
            gc.collect()
            torch.cuda.empty_cache()

## 7. 汇总与导出
没有 A/P 标注前，生成成功不代表丰富度提升。

In [ ]:
from collections import Counter

records = [
    Attempt.model_validate_json(p.read_text(encoding="utf-8"))
    for p in sorted(OUTPUT.glob("Q*__*__r*.json"))
]
print("记录数:", len(records), "/ 36")
print("状态:", dict(Counter(r.status for r in records)))
print("条件:", dict(Counter(r.condition for r in records)))
for record in records:
    print(
        record.question_id, record.condition, record.draw_index + 1, record.status, record.selected
    )
print("这些是已见案例的开发结果；未进行 A/P 标注，不构成 agent 优势结论。")

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive(
    str(RESULTS_ROOT / f"{RUN_LABEL}_results"),
    "zip",
    root_dir=RESULTS_ROOT,
    base_dir=RUN_LABEL,
)
files.download(archive)